# Acquisition des données reprise de la correction du TD3, puis adaptée au TD4

In [1]:
# Recharge automatiquement nos fichiers .py (Document, Author, Corpus)
# quand on les modifie, sans redémarrer le noyau
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
import html
import urllib.request
import urllib.parse

import xmltodict
import pandas as pd
from datetime import datetime

from Document import Document
from Author import Author

In [3]:
# la thématique choisie, à modifier librement
theme = "climate"
id2doc = {}  
id2aut = {}
identifiant = 0    

In [4]:

url_hn = "https://hn.algolia.com/api/v1/search?" + urllib.parse.urlencode(
    {"query": theme, "tags": "story", "hitsPerPage": 50})
print(url_hn)

with urllib.request.urlopen(url_hn) as reponse:
    d_hn = json.load(reponse)

https://hn.algolia.com/api/v1/search?query=climate&tags=story&hitsPerPage=50


In [5]:
print(d_hn.keys())           
print(d_hn["nbHits"])       
hits = d_hn["hits"]
print(len(hits))             
print(hits[0].keys())        

dict_keys(['exhaustive', 'exhaustiveNbHits', 'exhaustiveTypo', 'hits', 'hitsPerPage', 'nbHits', 'nbPages', 'page', 'params', 'processingTimeMS', 'processingTimingsMS', 'query', 'serverTimeMS'])
18475
50
dict_keys(['_highlightResult', '_tags', 'author', 'children', 'created_at', 'created_at_i', 'num_comments', 'objectID', 'points', 'story_id', 'title', 'updated_at', 'url'])


In [6]:

for hit in hits:
    texte = hit.get("story_text")
    if not texte:
        texte = hit.get("title", "")

    texte = html.unescape(texte).replace("<p>", " ").strip()
    texte = texte.replace("\n", " ")

    titre = hit.get("title", "")
    auteur = hit.get("author", "")
    date = datetime.strptime(hit["created_at"][:10], "%Y-%m-%d")
    url = hit.get("url")
    if not url:   # posts sans lien : on fabrique l'adresse du post
        url = "https://news.ycombinator.com/item?id=" + hit["objectID"]
        
    doc = Document(titre, auteur, date, url, texte)
    id2doc[identifiant] = doc

    if auteur not in id2aut:             
        id2aut[auteur] = Author(auteur)   
    id2aut[auteur].add(identifiant, doc)
    
    identifiant += 1

    

print("Documents Hacker News :", len(id2doc))

Documents Hacker News : 50


In [7]:
url_arxiv = "http://export.arxiv.org/api/query?" + urllib.parse.urlencode(
    {"search_query": "all:" + theme, "start": 0, "max_results": 50})
print(url_arxiv)

with urllib.request.urlopen(url_arxiv) as reponse:
    d_arxiv = xmltodict.parse(reponse.read())

http://export.arxiv.org/api/query?search_query=all%3Aclimate&start=0&max_results=50


In [8]:
print(d_arxiv.keys())               
print(d_arxiv["feed"].keys())        

entrees = d_arxiv["feed"].get("entry", [])
if isinstance(entrees, dict):
    entrees = [entrees]
print(len(entrees))
if entrees:                          
    print(entrees[0].keys())         

dict_keys(['feed'])
dict_keys(['@xmlns:opensearch', '@xmlns:arxiv', '@xmlns', 'id', 'title', 'updated', 'link', 'opensearch:itemsPerPage', 'opensearch:totalResults', 'opensearch:startIndex', 'entry'])
50
dict_keys(['id', 'title', 'updated', 'link', 'summary', 'category', 'published', 'arxiv:comment', 'arxiv:primary_category', 'author'])


In [9]:
for entree in entrees:
    texte = entree["summary"].replace("\n", " ").strip()

    # TD4 : les métadonnées
    titre = entree["title"].replace("\n", " ").strip()
    date = datetime.strptime(entree["published"][:10], "%Y-%m-%d")
    url = entree["id"]

    auteurs = entree["author"]
    if isinstance(auteurs, dict):     
        auteurs = [auteurs]
    auteur = auteurs[0]["name"]       

    # TD4 : on crée le Document et on le range (comme pour Hacker News)
    doc = Document(titre, auteur, date, url, texte)
    id2doc[identifiant] = doc

    if auteur not in id2aut:
        id2aut[auteur] = Author(auteur)
    id2aut[auteur].add(identifiant, doc)

    identifiant += 1

print("Taille du corpus :", len(id2doc))

Taille du corpus : 100


In [10]:
print(len(id2doc))     
print(id2doc[60])      
id2doc[60].afficher()  

100
 Deep Learning for Climate Model Output Statistics
 le titre du document est : Deep Learning for Climate Model Output Statistics,l'auteur est:Michael Steininger, la date: 2020-12-09 00:00:00,l'url :http://arxiv.org/abs/2012.10394v1 et le text correspodant est :            Climate models are an important tool for the assessment of prospective climate change effects but they suffer from systematic and representation errors, especially for precipitation. Model output statistics (MOS) reduce these errors by fitting the model output to observational data with machine learning. In this work, we explore the feasibility and potential of deep learning with convolutional neural networks (CNNs) for MOS. We propose the CNN architecture ConvMOS specifically designed for reducing errors in climate model outputs and apply it to the climate model REMO. Our results show a considerable reduction of errors and mostly improved performance compared to three commonly used MOS approaches.


In [11]:

#df.to_csv("corpus.csv", sep="\t", index=False)

In [12]:
#df = pd.read_csv("corpus.csv", sep="\t")
#df["texte"] = df["texte"].fillna("")   
#print(df.shape)
 

In [13]:
#print("Nombre de documents :", len(df))

In [15]:
#for texte in df["texte"]:
  #  mots = texte.split(" ")
   # phrases = texte.split(".")
    #print(len(mots), "mots,", len(phrases), "phrases")

In [17]:

#print("Avant filtrage :", len(df))
#df = df[df["texte"].str.len() >= 20]
#print("Après filtrage :", len(df))

In [18]:
# TD3 3.4 adapté au TD4 : une chaîne avec tous les textes du corpus
chaine = " ".join(doc.texte for doc in id2doc.values())
print(len(chaine))
print(chaine[:500])

63594
Climate change: IPCC report is 'code red for humanity' Climate activist arrested after ProtonMail provided his IP address White House eyes subsidies for nuclear plants to help meet climate targets DOE announces breakthrough in residential cold climate heat pump technology Websites hosting major US climate reports taken down Project Vesta – Mitigating climate change with green sand beaches Former NOAA employees built Climate.us to preserve climate data and resources WA, NY and CA Governors Announ


In [21]:
print(list(id2aut.keys())[:10])

['perfunctory', 'kdunglas', 'pseudolus', 'josephscott', 'geox', 'QuickToBan', 'benwerd', 'rbanffy', 'antouank', 'scrum-treats']


In [20]:
# 2.4 : statistiques sur un auteur choisi par l'utilisateur
nom = input("Nom de l'auteur : ")

if nom not in id2aut:
    print("Auteur inconnu.")
else:
    #notre objet auteur
    auteur = id2aut[nom]                       

    # Nombre de documents : déjà calculé par add()
    print("Nombre de documents :", auteur.ndoc)

    # Taille moyenne : on additionne la taille de chaque document
    total = 0
    for doc in auteur.production.values():     
        total += len(doc.texte)               
    moyenne = total / auteur.ndoc
    print("Taille moyenne :", round(moyenne, 1), "caractères")

Nom de l'auteur :  scrum-treats


Nombre de documents : 1
Taille moyenne : 77.0 caractères
